# Isolation Forest for Network Traffic Anomaly Detection


## 1.  Initial Setup

In [78]:
import numpy as np
import pandas as pd
import random

from typing import List, Dict

from sklearn.ensemble import IsolationForest as SklearnIsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import confusion_matrix, precision_score, recall_score

In [79]:
train_df: pd.DataFrame = pd.read_csv("../data/UNSW_NB15_training-set.csv")
test_df: pd.DataFrame = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

In [ ]:
SAMPLE_SIZE: int = 256
N_TREES: int = 10
MAX_DEPTH: int = 8

FEATURES: List[str] = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes",
    "rate", "sload", "dload", "sinpkt", "dinpkt"
]

FEATURES_V2: List [str] = FEATURES + [
    "ct_dst_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_srv_dst",
]

## 2. Isolation Tree model

In [81]:
class Isolation_Tree:
    def __init__(self, df: pd.DataFrame, max_depth: int):
        self.df: pd.DataFrame = df
        self.max_depth: int = max_depth
        self.root: Dict[str, str]  = self.build_tree(self.df)

    def random_feature(self, node_data: pd.DataFrame) -> str:
        """
        Picks a random feature for the isolation tree.
        """
        features: List[str] = [feature for feature in node_data.columns if node_data[feature].nunique() > 1]
        rand_features: str = random.choice(features)

        return rand_features

    def split_values(self, feature: str, data: pd.DataFrame) -> float:
        """
        Splits the instances of a node randomly.
        """
        lowest_int: np.float64  = data[feature].min()
        highest_int: np.float64 = data[feature].max()
        rand_split: float = random.uniform(lowest_int, highest_int)

        return rand_split

    def partition_data(self, node_data: pd.DataFrame, select_feature: str, split_value: float):
        """
        Partitions the data of the selected parent node into two child nodes.
        """
        left_node: pd.DataFrame = node_data[node_data[select_feature] < split_value]
        right_node: pd.DataFrame = node_data[node_data[select_feature] >= split_value]

        return left_node, right_node
                        
    def build_tree(self, node_data: pd.DataFrame, depth: int = 0) -> Dict[str, str]:
        """
        Build a single isolation tree based on a randomly selected feature.
        """
        if (
            depth >= self.max_depth 
            or len(node_data) <= 1  
            or all(node_data[col].nunique() <= 1 for col in node_data.columns)
            ):
            return {"leaf_size": len(node_data)}

        rand_feature: str = self.random_feature(node_data)      
        rand_split: float = self.split_values(rand_feature, node_data)
        left_node, right_node = self.partition_data(node_data, rand_feature, rand_split)

        return {
        "feature": rand_feature,
        "split_value": rand_split,
        "left_node": self.build_tree(left_node, depth + 1),
        "right_node": self.build_tree(right_node, depth + 1),
                }

    def return_path_length(self, row: pd.Series) -> float:
        """
        Returns path length of a row.
        """
        node: Dict[str, str] = self.root
        path_length: int = 0

        while "leaf_size" not in node:
            feature: str = node["feature"]
            split_value: int = node["split_value"]

            if row[feature] < split_value:
                node = node["left_node"]
            else:
                node = node["right_node"]
            
            path_length += 1

        leaf_size: int = node["leaf_size"]

        return path_length + self.expected_extra_length(leaf_size)

    def expected_extra_length(self, n: int) -> float:
        """
        Calculates the extra length of a path length based on leaf nodes size.
        """
        if n <= 1:
            return 0

        if n == 2:
            return 1

        harmonic: float = sum(1 / i for i in range(1, n))
        expected_length: float = (2 * harmonic) - (2 * (n - 1)/ n)
        
        return expected_length

## 3. Isolation Forest model

In [82]:
class Isolation_Forest:
    def __init__(self, df: pd.DataFrame, sample_size: int, n_estimators: int, max_depth: int):
        self.df: pd.DataFrame = df
        self.sample_size: int = sample_size
        self.n_estimators: int = n_estimators
        self.trees: List[Isolation_Tree] = self.build_ensemble(df, max_depth)

    def random_sample(self, data_set: pd.DataFrame) -> List[pd.DataFrame]:
        """
        Returns a random sample of a dataset for isolation.
        """
        sample_size: int = self.sample_size
        sample_list: List[pd.DataFrame] = []

        for _ in range(0, self.n_estimators):
            samples: List[pd.Series] = []
            df_shuffled: pd.DataFrame = data_set.sample(frac=1,
                                                        random_state=42 + len(sample_list)
                                                        ).reset_index(drop=True)
            for j in range(0, sample_size):
                samples.append(df_shuffled.iloc[j])

            df: pd.DataFrame = pd.DataFrame(samples)
            sample_list.append(df)

        return sample_list

    def build_ensemble(self, data_set: pd.DataFrame, max_depth: int) -> List[Isolation_Tree]:
        """
        Builds an Isolation Tree for each sampled dataset.
        """
        sample_list: List[pd.DataFrame] = self.random_sample(data_set)
        tree_list: List[Isolation_Tree] = []

        for sample in sample_list:
            isolation_tree: Isolation_Tree = Isolation_Tree(sample, max_depth)
            tree_list.append(isolation_tree)

        return tree_list

    def average_path_length(self, row: pd.Series) -> float:
        """
        Return the average path length of the Isolation Tree forest.
        """
        lengths: List[float] = []
    
        for tree in self.trees:
            lengths.append(tree.return_path_length(row))

        average: float = sum(lengths) / len(lengths)
        
        return average

    def anomaly_score(self, row: pd.Series) -> float:
        """
        Converts average into an anomaly score.
        """
        c_sample: float = self.trees[0].expected_extra_length(self.sample_size)
        average: float = self.average_path_length(row)

        ratio: float = 2 ** (-average / c_sample)

        return ratio

# 4. Testing and Training - V1

In [ ]:
fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"],
)

X_fit: pd.DataFrame = fit_df.loc[fit_df["label"] == 0, FEATURES].sample(
    n=2_000, random_state=42
)

validation: pd.DataFrame = pd.concat([
    val_df.loc[val_df["label"] == 0].sample(n=500, random_state=42),
    val_df.loc[val_df["label"] == 1].sample(n=500, random_state=42),
]).sample(frac=1, random_state=42)

X_val: pd.DataFrame = validation[FEATURES]
y_val: pd.DataFrame = validation["label"]

In [ ]:
random.seed(42)

my_forest: Isolation_Forest = Isolation_Forest(
    X_fit,
    sample_size=SAMPLE_SIZE,
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
)

my_scores: pd.Series = X_val.apply(my_forest.anomaly_score, axis=1)

In [ ]:
sklearn_forest: SklearnIsolationForest = SklearnIsolationForest(
    n_estimators=N_TREES,
    max_samples=SAMPLE_SIZE,
    contamination="auto",
    random_state=42,
)
sklearn_forest.fit(X_fit)

sklearn_scores: pd.Series = pd.Series(
    -sklearn_forest.score_samples(X_val),
    index=X_val.index,
)

In [ ]:
results: pd.DataFrame = pd.DataFrame({
    "label": y_val,
    "my_score": my_scores,
    "sklearn_score": sklearn_scores,
})

print("Scores by true label (0 = benign, 1 = attack):")
print(results.groupby("label")[["my_score", "sklearn_score"]].agg(
    ["mean", "median", "min", "max"]
))

print("\nRanking metrics:")
for name in ["my_score", "sklearn_score"]:
    print(
        name,
        "ROC-AUC:", round(roc_auc_score(results["label"], results[name]), 3),
        "Average precision:", round(
            average_precision_score(results["label"], results[name]), 3
        ),
    )

print("\nHighest-scoring rows in your model:")
print(results.sort_values("my_score", ascending=False).head(10))

Scores by true label (0 = benign, 1 = attack):
       my_score                               sklearn_score            \
           mean    median       min       max          mean    median   
label                                                                   
0      0.389298  0.353284  0.320114  0.724911      0.390263  0.367801   
1      0.394971  0.382923  0.325083  0.674456      0.418498  0.398950   

                          
            min      max  
label                     
0      0.329884  0.70793  
1      0.336060  0.70435  

Ranking metrics:
my_score ROC-AUC: 0.601 Average precision: 0.547
sklearn_score ROC-AUC: 0.657 Average precision: 0.632

Highest-scoring rows in your model:
        label  my_score  sklearn_score
34716       0  0.724911       0.645457
16928       0  0.710351       0.632488
3158        0  0.693805       0.703156
9612        0  0.689128       0.707930
25712       0  0.677086       0.628157
49800       1  0.674456       0.684622
8080        0  0.6391

In [87]:
results["attack_cat"] = validation["attack_cat"]

print(
    results.groupby("attack_cat")[["my_score", "sklearn_score"]]
    .agg(["count", "mean", "median"])
    .sort_values(("my_score", "mean"), ascending=False)
)

               my_score                     sklearn_score                    
                  count      mean    median         count      mean    median
attack_cat                                                                   
Shellcode             4  0.407572  0.395362             4  0.449519  0.378467
DoS                  52  0.401528  0.381200            52  0.423591  0.391327
Exploits            127  0.400865  0.381200           127  0.433802  0.391327
Generic             173  0.398641  0.382923           173  0.421833  0.398950
Fuzzers              77  0.397670  0.365564            77  0.415870  0.389601
Normal              500  0.389298  0.353284           500  0.390263  0.367801
Backdoor              4  0.388712  0.382062             4  0.393232  0.391327
Analysis             16  0.379780  0.381200            16  0.389296  0.391327
Reconnaissance       47  0.358493  0.344627            47  0.372995  0.367204


## 5. Testing and Training - V2

In [96]:
# Train on benign flows only.
X_fit2: pd.DataFrame = fit_df.loc[fit_df["label"] == 0, FEATURES_V2].sample(
    n=2_000, random_state=42
)

X_val2: pd.DataFrame = validation[FEATURES_V2]

In [ ]:
random.seed(42)

my_forest: Isolation_Forest = Isolation_Forest(
    X_fit2,
    sample_size=SAMPLE_SIZE,
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
)

my_scores: pd.Series = X_val2.apply(my_forest.anomaly_score, axis=1)

results: pd.DataFrame = pd.DataFrame({
    "label": y_val,
    "anomaly_score": my_scores,
})

In [ ]:
sklearn_forest: SklearnIsolationForest = SklearnIsolationForest(
    n_estimators=N_TREES,
    max_samples=SAMPLE_SIZE,
    contamination="auto",
    random_state=42,
)
sklearn_forest.fit(X_fit2)

sklearn_scores: pd.Series = pd.Series(
    -sklearn_forest.score_samples(X_val2),
    index=X_val.index,
)

In [ ]:
results: pd.DataFrame = pd.DataFrame({
    "label": y_val,
    "my_score": my_scores,
    "sklearn_score": sklearn_scores,
})

print("Scores by true label (0 = benign, 1 = attack):")
print(results.groupby("label")[["my_score", "sklearn_score"]].agg(
    ["mean", "median", "min", "max"]
))

print("\nRanking metrics:")
for name in ["my_score", "sklearn_score"]:
    print(
        name,
        "ROC-AUC:", round(roc_auc_score(results["label"], results[name]), 3),
        "Average precision:", round(
            average_precision_score(results["label"], results[name]), 3
        ),
    )

print("\nHighest-scoring rows in your model:")
print(results.sort_values("my_score", ascending=False).head(10))

Scores by true label (0 = benign, 1 = attack):
       my_score                               sklearn_score            \
           mean    median       min       max          mean    median   
label                                                                   
0      0.371782  0.352901  0.313356  0.648658      0.392608  0.382685   
1      0.462444  0.446182  0.313356  0.701914      0.490354  0.466895   

                           
            min       max  
label                      
0      0.328186  0.657934  
1      0.328186  0.708857  

Ranking metrics:
my_score ROC-AUC: 0.735 Average precision: 0.771
sklearn_score ROC-AUC: 0.707 Average precision: 0.779

Highest-scoring rows in your model:
        label  my_score  sklearn_score
164471      1  0.701914       0.679153
147239      1  0.701914       0.661019
144503      1  0.701914       0.694614
126454      1  0.687816       0.661019
174992      1  0.687816       0.679153
133863      1  0.687816       0.661019
135823      1  0

## 6. Evaluation

In [99]:
# Choose thresholds using benign VALIDATION rows only.
my_threshold: float = results.loc[
    results["label"] == 0, "my_score"
].quantile(0.95)

sklearn_threshold: float = results.loc[
    results["label"] == 0, "sklearn_score"
].quantile(0.95)

for name, score_column, threshold in [
    ("Mine", "my_score", my_threshold),
    ("scikit-learn", "sklearn_score", sklearn_threshold),
]:
    predicted_attack: pd.Series = (results[score_column] >= threshold).astype(int)

    print(f"\n{name}")
    print("Threshold:", threshold)
    print("Confusion matrix:\n", confusion_matrix(results["label"], predicted_attack))
    print("Precision:", precision_score(results["label"], predicted_attack))
    print("Recall:", recall_score(results["label"], predicted_attack))


Mine
Threshold: 0.5000305973748529
Confusion matrix:
 [[475  25]
 [309 191]]
Precision: 0.8842592592592593
Recall: 0.382

scikit-learn
Threshold: 0.5274346271837786
Confusion matrix:
 [[475  25]
 [280 220]]
Precision: 0.8979591836734694
Recall: 0.44


In [ ]:
X_test: pd.DataFrame = test_df[FEATURES_V2]
y_test: pd.DataFrame = test_df["label"]

my_test_scores: pd.Series = X_test.apply(my_forest.anomaly_score, axis=1)
sklearn_test_scores: np.array = -sklearn_forest.score_samples(X_test)

my_test_alerts: pd.Series = (my_test_scores >= my_threshold).astype(int)
sklearn_test_alerts: np.array = (sklearn_test_scores >= sklearn_threshold).astype(int)

for name, scores, alerts in [
    ("Mine", my_test_scores, my_test_alerts),
    ("scikit-learn", sklearn_test_scores, sklearn_test_alerts),
]:
    print(f"\n{name}")
    print("ROC-AUC:", roc_auc_score(y_test, scores))
    print("Average precision:", average_precision_score(y_test, scores))
    print("Confusion matrix:\n", confusion_matrix(y_test, alerts))
    print("Precision:", precision_score(y_test, alerts, zero_division=0))
    print("Recall:", recall_score(y_test, alerts, zero_division=0))